# Explainable Hybrid Fraud Detection System

**Notebook:** End-to-end walkthrough of all 10 phases.

Datasets:
- [Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
- [PaySim](https://www.kaggle.com/datasets/ealaxi/paysim1)

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## Phase 1: Data Ingestion & EDA

In [ ]:
from data.ingest import DataIngestion

# Load Credit Card dataset (synthetic if CSV not found)
ingestion = DataIngestion(dataset='creditcard')
df = ingestion.load()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# EDA Report
eda = ingestion.eda()
print(f"Fraud Rate: {eda['fraud_rate_pct']}%")
print(f"Missing Values: {sum(eda['missing_values'].values())}")
print(f"Duplicates: {eda['duplicates']}")

# Class distribution
dist = eda['class_distribution']
fig = go.Figure(go.Pie(
    labels=['Normal', 'Fraud'],
    values=[dist.get('0', dist.get(0, 0)), dist.get('1', dist.get(1, 0))],
    hole=0.4,
    marker_colors=['#2196F3', '#F44336']
))
fig.update_layout(title='Class Distribution')
fig.show()

In [ ]:
# Clean and split
ingestion.clean()
X_train_raw, X_test_raw, y_train, y_test = ingestion.split(strategy='smote')
print(f'Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
print(f'Train class dist: {dict(pd.Series(y_train).value_counts())}')

## Phase 2: Feature Engineering

In [ ]:
from pipelines.feature_pipeline import get_feature_pipeline

pipeline = get_feature_pipeline('creditcard')
X_train = pipeline.fit_transform(X_train_raw, y_train)
X_test = pipeline.transform(X_test_raw)
print(f'Engineered features: {X_train.shape[1]} (was {X_train_raw.shape[1]})')

## Phase 3: ML Model Training

In [ ]:
from models.trainer import ModelTrainer

trainer = ModelTrainer(dataset='creditcard')
results = trainer.train_all(X_train, y_train, X_test, y_test)

comparison = trainer.get_comparison_df()
print('\nModel Comparison:')
print(comparison.to_string())
print(f'\nBest Model: {trainer.best_model_name}')

In [ ]:
# Visualize comparison
fig = px.bar(
    comparison.reset_index().melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']),
    x='variable', y='value', color='Model', barmode='group',
    title='Model Performance Comparison',
    labels={'variable': 'Metric', 'value': 'Score'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

## Phase 4: Anomaly Detection

In [ ]:
from models.anomaly_detector import AnomalyDetector

detector = AnomalyDetector()
detector.fit(X_train)
anomaly_metrics = detector.evaluate(X_test, y_test)
print('Isolation Forest Metrics:')
for k, v in anomaly_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

## Phase 5: Hybrid Engine

In [ ]:
from models.hybrid_engine import HybridFraudEngine

engine = HybridFraudEngine(
    supervised_model=trainer.best_model,
    anomaly_detector=detector.model,
)

# Predict on test set
predictions = engine.predict_full(X_test[:20])
pred_df = pd.DataFrame(predictions)
pred_df['actual'] = list(y_test[:20])
print(pred_df[['fraud_probability', 'anomaly_score', 'risk_score', 'risk_category', 'actual']].to_string())

In [ ]:
# Risk score distribution
all_preds = engine.predict_full(X_test)
all_df = pd.DataFrame(all_preds)

fig = px.histogram(all_df, x='risk_score', color='risk_category',
    title='Risk Score Distribution',
    color_discrete_map={'Low': '#28a745', 'Medium': '#ffc107', 'High': '#fd7e14', 'Critical': '#dc3545'}
)
fig.show()

## Phase 6: SHAP Explainability

In [ ]:
from models.explainer import FraudExplainer

explainer = FraudExplainer(model=trainer.best_model)
background = X_test[:200]
explainer.build_explainer(background)

# Global importance
importance_df = explainer.global_importance(background)
print('Top 10 Features by SHAP Importance:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Visualize global importance
fig = px.bar(
    importance_df.head(15),
    x='importance', y='feature', orientation='h',
    title='Global SHAP Feature Importance',
    color='importance', color_continuous_scale='RdYlGn_r'
)
fig.update_layout(yaxis={'autorange': 'reversed'}, height=500)
fig.show()

In [ ]:
# Local explanation for a fraud case
fraud_idx = np.where(np.array(y_test) == 1)[0]
if len(fraud_idx) > 0:
    idx = fraud_idx[0]
    local = explainer.local_explanation(X_test[idx:idx+1])
    print(f'\nFraud Probability: {trainer.best_model.predict_proba(X_test[idx:idx+1])[0,1]*100:.1f}%')
    print('\nTop Reasons:')
    for reason in local['reasons']:
        print(f'  - {reason}')

## Phase 7: Research Experiments

In [ ]:
from experiments.runner import ExperimentRunner

runner = ExperimentRunner(X_train, y_train, X_test, y_test)
exp_results = runner.run_all()

exp_df = pd.DataFrame(exp_results)
print(f'Completed {len(exp_df)} experiments')
exp_df.head(10)

In [ ]:
# Visualize experiment results
fig = px.bar(
    exp_df.melt(id_vars=['experiment', 'name'], value_vars=['f1_score', 'roc_auc']),
    x='name', y='value', color='variable', barmode='group',
    facet_col='experiment', title='Research Experiment Results',
    labels={'name': 'Config', 'value': 'Score'}
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

## Summary

All 10 phases completed successfully!

- Models saved to `models/`
- Reports saved to `reports/`
- Experiments saved to `experiments/`

**Next Steps:**
- Start API: `uvicorn api.main:app --reload`
- Start Dashboard: `streamlit run dashboard/app.py`
- View MLflow: `mlflow ui`